# Task 1 — Fine-tuning a Pretrained Text-to-Image Model (Stable Diffusion + LoRA)

**Goal:** Refine a pretrained text-to-image model (Stable Diffusion) on a
custom, small image dataset.

**Approach — LoRA, not full fine-tuning:** Full fine-tuning of Stable
Diffusion updates ~1 billion parameters and needs far more VRAM/time than a
free-tier Colab T4 (16GB) reasonably allows. **LoRA (Low-Rank Adaptation)**
freezes the entire pretrained model and injects small trainable low-rank
matrices into the UNet's attention layers — typically well under 1% of the
full parameter count. This is the standard, widely-used approach for
fine-tuning diffusion models on consumer/free-tier GPUs, not a shortcut that
undersells the task.

**Custom dataset:** a small subset of Oxford-102 Flowers (2–3 classes,
~40–60 images total) — consistent with the rest of this project, and a
realistic size for LoRA, which is specifically designed to work well with
small datasets.

**Base model:** `runwayml/stable-diffusion-v1-5` via HuggingFace `diffusers`,
with LoRA layers managed through HuggingFace `peft`.

## 1. Setup

**Important:** after installing, go to **Runtime → Restart session**, then
continue from the next cell. This avoids version-conflict issues between
freshly installed `diffusers`/`peft` and whatever Colab pre-installed.

In [ ]:
!pip install -q diffusers[torch] transformers accelerate peft --upgrade
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121 --force-reinstall
print('Installed. Now: Runtime > Restart session, then continue from the next cell.')

In [ ]:
import os
import shutil
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from torchvision.datasets import Flowers102

from diffusers import StableDiffusionPipeline, UNet2DConditionModel, DDPMScheduler, AutoencoderKL
from transformers import CLIPTextModel, CLIPTokenizer
from peft import LoraConfig, get_peft_model

import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
assert device.type == 'cuda', 'This notebook needs a GPU. Runtime > Change runtime type > T4 GPU.'

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

## 2. Build the small custom dataset

We pick 2–3 visually distinctive flower classes from Oxford-102 and use
~15–20 images per class (dataset downloaded to local disk, as in all
previous tasks). Images are resized to 512x512, the resolution Stable
Diffusion v1.5 was trained at.

In [ ]:
DATA_ROOT = '/content/data'
os.makedirs(DATA_ROOT, exist_ok=True)

# sunflower (=50), rose (=72), water lily (=69) -- visually distinctive classes
SELECTED_CLASSES = {50: 'sunflower', 72: 'rose', 69: 'water lily'}
IMAGES_PER_CLASS = 15

raw_train = Flowers102(root=DATA_ROOT, split='train', download=True)
raw_val = Flowers102(root=DATA_ROOT, split='val', download=True)
combined = torch.utils.data.ConcatDataset([raw_train, raw_val])

# Collect indices belonging to our selected classes
selected_items = []  # list of (PIL image, caption)
counts = {c: 0 for c in SELECTED_CLASSES}

for img, label in combined:
    if label in SELECTED_CLASSES and counts[label] < IMAGES_PER_CLASS:
        caption = f'a photo of a {SELECTED_CLASSES[label]}'
        selected_items.append((img, caption))
        counts[label] += 1
    if all(c >= IMAGES_PER_CLASS for c in counts.values()):
        break

print('Collected images per class:', counts)
print('Total custom dataset size:', len(selected_items))

In [ ]:
# Visual check of the custom dataset
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for ax, (img, cap) in zip(axes, random.sample(selected_items, 6)):
    ax.imshow(img)
    ax.set_title(cap, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.savefig('/content/custom_dataset_sample.png', dpi=150)
plt.show()

## 3. Load the pretrained Stable Diffusion components

We load the pipeline's individual components (VAE, text encoder, tokenizer,
UNet, noise scheduler) rather than the high-level pipeline object, since LoRA
training needs direct access to the UNet.

In [ ]:
MODEL_NAME = 'runwayml/stable-diffusion-v1-5'

tokenizer = CLIPTokenizer.from_pretrained(MODEL_NAME, subfolder='tokenizer')
text_encoder = CLIPTextModel.from_pretrained(MODEL_NAME, subfolder='text_encoder').to(device)
vae = AutoencoderKL.from_pretrained(MODEL_NAME, subfolder='vae').to(device)
unet = UNet2DConditionModel.from_pretrained(MODEL_NAME, subfolder='unet').to(device)
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_NAME, subfolder='scheduler')

# Freeze everything except the LoRA layers we're about to add
vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

vae.eval()
text_encoder.eval()
print('Base model components loaded and frozen.')

## 4. Inject LoRA layers into the UNet's attention modules

We use `peft`'s `LoraConfig`, targeting the UNet's cross-attention and
self-attention projection layers (`to_k`, `to_q`, `to_v`, `to_out.0`) — the
standard target modules for Stable Diffusion LoRA.

In [ ]:
lora_config = LoraConfig(
    r=4,                      # rank of the low-rank matrices — small = fewer trainable params
    lora_alpha=4,
    target_modules=['to_k', 'to_q', 'to_v', 'to_out.0'],
    lora_dropout=0.0,
)

unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

## 5. Dataset and DataLoader for training

Preprocesses images (resize/normalize to what the VAE expects) and tokenizes
captions.

In [ ]:
RESOLUTION = 512

image_transform = T.Compose([
    T.Resize((RESOLUTION, RESOLUTION)),
    T.ToTensor(),
    T.Normalize([0.5], [0.5]),  # VAE expects inputs in [-1, 1]
])

class LoraFineTuneDataset(Dataset):
    def __init__(self, items, tokenizer, transform):
        self.items = items
        self.tokenizer = tokenizer
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img, caption = self.items[idx]
        pixel_values = self.transform(img.convert('RGB'))
        input_ids = self.tokenizer(
            caption, padding='max_length', truncation=True,
            max_length=self.tokenizer.model_max_length, return_tensors='pt'
        ).input_ids[0]
        return {'pixel_values': pixel_values, 'input_ids': input_ids}

train_dataset = LoraFineTuneDataset(selected_items, tokenizer, image_transform)
train_dataloader = DataLoader(train_dataset, batch_size=2, shuffle=True)

print(f'Training dataset size: {len(train_dataset)}')
print(f'Batches per epoch: {len(train_dataloader)}')

## 6. Generate baseline images BEFORE fine-tuning

Important for showing the effect of fine-tuning: we generate images from the
**unmodified base model** first, so we have a clear before/after comparison.

In [ ]:
VALIDATION_PROMPTS = [
    'a photo of a sunflower',
    'a photo of a rose',
    'a photo of a water lily',
]

def build_pipeline_from_components():
    """Assemble a StableDiffusionPipeline from our current (possibly LoRA-adapted) components."""
    pipe = StableDiffusionPipeline.from_pretrained(
        MODEL_NAME,
        vae=vae,
        text_encoder=text_encoder,
        tokenizer=tokenizer,
        unet=unet,
        safety_checker=None,
    ).to(device)
    return pipe

print('Generating BEFORE (base model, no fine-tuning) images...')
pipe = build_pipeline_from_components()
pipe.set_progress_bar_config(disable=True)

generator = torch.Generator(device=device).manual_seed(123)
before_images = [pipe(p, num_inference_steps=25, generator=generator).images[0] for p in VALIDATION_PROMPTS]

fig, axes = plt.subplots(1, len(VALIDATION_PROMPTS), figsize=(15, 5))
for ax, img, prompt in zip(axes, before_images, VALIDATION_PROMPTS):
    ax.imshow(img)
    ax.set_title(f'BEFORE:\n"{prompt}"', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.savefig('/content/before_finetuning.png', dpi=150)
plt.show()

del pipe
torch.cuda.empty_cache()

## 7. LoRA training loop

Standard diffusion training objective: add noise to an image's latent
representation, then train the UNet (through its LoRA layers only) to
predict that noise given the text conditioning — this is exactly how
Stable Diffusion itself was originally trained, just with almost everything
frozen except the small LoRA matrices.

**Time estimate:** with ~45 images, batch size 2, this is intentionally kept
to a small number of epochs (LoRA converges fast on tiny datasets and
overfitting is a real risk if pushed too far). Expect roughly 15–25 minutes
total on a T4.

In [ ]:
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4

optimizer = torch.optim.AdamW(
    [p for p in unet.parameters() if p.requires_grad],
    lr=LEARNING_RATE
)

unet.train()
losses = []

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_loss = 0.0
    for batch in train_dataloader:
        pixel_values = batch['pixel_values'].to(device, dtype=torch.float32)
        input_ids = batch['input_ids'].to(device)

        # Encode images to latent space via the frozen VAE
        with torch.no_grad():
            latents = vae.encode(pixel_values).latent_dist.sample()
            latents = latents * vae.config.scaling_factor

        # Sample random noise and a random timestep per image
        noise = torch.randn_like(latents)
        bsz = latents.shape[0]
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        # Get text embeddings (frozen text encoder)
        with torch.no_grad():
            encoder_hidden_states = text_encoder(input_ids)[0]

        # Predict the noise residual (this is the only part with trainable LoRA params)
        noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
        loss = F.mse_loss(noise_pred, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_dataloader)
    losses.append(avg_loss)
    print(f'Epoch [{epoch}/{NUM_EPOCHS}]  Loss: {avg_loss:.4f}')

print('\nLoRA fine-tuning complete.')

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('MSE denoising loss')
plt.title('LoRA Fine-tuning Loss')
plt.tight_layout()
plt.savefig('/content/lora_training_loss.png', dpi=150)
plt.show()

## 8. Save the LoRA weights

One of the big advantages of LoRA: the saved file is tiny (a few MB) compared
to the full ~4GB Stable Diffusion checkpoint, since we're only saving the
small adapter matrices.

In [ ]:
LORA_SAVE_DIR = '/content/lora_weights'
os.makedirs(LORA_SAVE_DIR, exist_ok=True)
unet.save_pretrained(LORA_SAVE_DIR)

size_mb = sum(
    os.path.getsize(os.path.join(LORA_SAVE_DIR, f))
    for f in os.listdir(LORA_SAVE_DIR)
) / (1024 * 1024)
print(f'Saved LoRA weights to {LORA_SAVE_DIR} ({size_mb:.1f} MB total)')

## 9. Generate images AFTER fine-tuning — the key comparison

Same prompts, same seed as the "before" images — any difference we see is
attributable to the LoRA fine-tuning.

In [ ]:
unet.eval()
pipe = build_pipeline_from_components()
pipe.set_progress_bar_config(disable=True)

generator = torch.Generator(device=device).manual_seed(123)  # same seed as before
after_images = [pipe(p, num_inference_steps=25, generator=generator).images[0] for p in VALIDATION_PROMPTS]

fig, axes = plt.subplots(1, len(VALIDATION_PROMPTS), figsize=(15, 5))
for ax, img, prompt in zip(axes, after_images, VALIDATION_PROMPTS):
    ax.imshow(img)
    ax.set_title(f'AFTER:\n"{prompt}"', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.savefig('/content/after_finetuning.png', dpi=150)
plt.show()

In [ ]:
# Side-by-side before/after grid — the clearest illustration for the report
fig, axes = plt.subplots(2, len(VALIDATION_PROMPTS), figsize=(15, 10))
for col, prompt in enumerate(VALIDATION_PROMPTS):
    axes[0, col].imshow(before_images[col])
    axes[0, col].set_title(f'BEFORE: "{prompt}"', fontsize=10)
    axes[0, col].axis('off')
    axes[1, col].imshow(after_images[col])
    axes[1, col].set_title(f'AFTER: "{prompt}"', fontsize=10)
    axes[1, col].axis('off')
plt.tight_layout()
plt.savefig('/content/before_after_comparison.png', dpi=150)
plt.show()

## 10. Copy results to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/elevance-skills/task1_finetune_diffusion'
os.makedirs(DRIVE_DIR, exist_ok=True)

shutil.copytree(LORA_SAVE_DIR, os.path.join(DRIVE_DIR, 'lora_weights'), dirs_exist_ok=True)
for fname in ['before_finetuning.png', 'after_finetuning.png', 'before_after_comparison.png',
              'lora_training_loss.png', 'custom_dataset_sample.png']:
    src = os.path.join('/content', fname)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(DRIVE_DIR, fname))

print('Copied LoRA weights and result images to:', DRIVE_DIR)
print(os.listdir(DRIVE_DIR))

## 11. Summary of findings

Fill this in after training, then copy into `NOTES.md` and today's daily log:
- Did the after-fine-tuning images visibly shift toward your custom dataset's
  style/subject compared to before?
- How did the training loss behave — did it decrease steadily?
- With only ~45 images and 15 epochs, would you expect strong or subtle
  changes? Is that what you observed?
- What would you try next (more images, more epochs, higher LoRA rank) to
  push the effect further?